<a href="https://colab.research.google.com/github/OdysseusPolymetis/atelier_humanistica2026/blob/main/5_rag_scaife_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG grec ancien-français, avec Qwen

Ce notebook construit un pipeline RAG (Retrieval Augmented Generation) sur des textes grecs anciens :

1. **Corpus grec** récupéré depuis github (Homère, Thucydide, Platon…).
2. **Paires alignées grec–français** (fichier `train.csv`) utilisées comme « pont bilingue » pour passer d'une question française au corpus grec.
3. **Trois stratégies de retrieval** comparées :
   - baseline directe français → grec avec un modèle multilingue (LaBSE) ;
   - pont bilingue : on cherche d'abord les paires françaises les plus proches, puis on encode leurs équivalents grecs comme requête vers le corpus grec ;
   - recherche directe dans les paires alignées (utile comme contrôle).
4. **Génération finale avec Qwen** : on passe les passages grecs récupérés (et l'indice français quand il existe) à Qwen, qui répond en français avec des citations `[P1]`, `[P2]`…

## 1. Installation

À exécuter une seule fois. On installe :

- `sentence-transformers`, `faiss-cpu` pour le retrieval ;
- `transformers`, `accelerate`, `bitsandbytes` pour Qwen avec quantization 4 bits.

In [1]:
!pip install -q \
    pandas numpy scikit-learn tqdm requests beautifulsoup4 \
    sentence-transformers faiss-cpu \
    "transformers>=4.45" accelerate bitsandbytes

## 2. Imports et configuration

In [2]:
import os
import re
import time
import random
import textwrap
from pathlib import Path
from urllib.parse import quote, unquote, urljoin

import numpy as np
import pandas as pd
import faiss
import requests
import torch

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device pour embeddings :", DEVICE)
if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

USE_FAISS_GPU_IF_AVAILABLE = True

ALIGNED_PAIRS_PATH = "train.csv"
GREEK_CORPUS_PATH  = None
GITHUB_CACHE_PATH  = "github_greek_corpus.csv"
SCAIFE_CACHE_PATH  = "scaife_greek_corpus.csv"

USE_GITHUB_XML_FIRST          = True
GITHUB_GROUP_TARGET_WORDS     = 200

USE_SCAIFE_ATLAS_CORPUS       = False
SCAIFE_MAX_PASSAGES_PER_TEXT  = 60
SCAIFE_SLEEP                  = 0.15

SCAIFE_EDITIONS = [
    {"edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2", "author": "Homerus",    "title": "Ilias",     "language": "grc"},
    {"edition_urn": "urn:cts:greekLit:tlg0012.tlg002.perseus-grc2", "author": "Homerus",    "title": "Odyssea",   "language": "grc"},
    {"edition_urn": "urn:cts:greekLit:tlg0003.tlg001.perseus-grc2", "author": "Thucydides", "title": "Historiae", "language": "grc"},
    {"edition_urn": "urn:cts:greekLit:tlg0059.tlg030.perseus-grc2", "author": "Plato",      "title": "Respublica","language": "grc"},
]

GREEK_PAIR_COLUMN  = "greek"
FRENCH_PAIR_COLUMN = "french"

GROUP_SIZE_FOR_RAG   = 8
CHUNK_SIZE_WORDS     = 120
CHUNK_OVERLAP_WORDS  = 30
MAX_CORPUS_CHUNKS    = None
MAX_ANCHOR_PAIRS     = None

CROSSLINGUAL_MODEL_NAME = "sentence-transformers/LaBSE"
GREEK_MODEL_NAME        = "bowphs/SPhilBerta"
BATCH_SIZE              = 32
ANCHOR_K                = 3
TOP_K                   = 5

QWEN_MODEL_NAME      = "Qwen/Qwen2.5-3B-Instruct"
QWEN_LOAD_IN_4BIT    = True
QWEN_MAX_NEW_TOKENS  = 512
QWEN_TEMPERATURE     = 0.2
QWEN_TOP_P           = 0.9

Device pour embeddings : cuda
GPU : Tesla T4


## 3. Fonctions utilitaires

In [3]:
def find_existing_path(path):
    if path is None:
        return None
    for candidate in [Path(path), Path("/content") / Path(path).name, Path("/mnt/data") / Path(path).name]:
        if candidate.exists():
            return candidate
    return None


def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_words(text, chunk_size=120, overlap=30):
    words = clean_text(text).split()
    if len(words) <= chunk_size:
        return [clean_text(text)] if clean_text(text) else []
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = max(0, end - overlap)
    return chunks


def maybe_limit_df(df, max_rows, random_state=RANDOM_STATE):
    if max_rows is None or len(df) <= max_rows:
        return df.reset_index(drop=True)
    return df.sample(max_rows, random_state=random_state).reset_index(drop=True)


def l2_normalize_vector(x):
    x = np.asarray(x, dtype="float32")
    norm = np.linalg.norm(x)
    return x if norm == 0 else x / norm


def build_faiss_ip_index(embeddings):
    embeddings = np.asarray(embeddings, dtype="float32")
    cpu_index = faiss.IndexFlatIP(embeddings.shape[1])
    cpu_index.add(embeddings)
    if USE_FAISS_GPU_IF_AVAILABLE:
        try:
            n_gpus = faiss.get_num_gpus() if hasattr(faiss, "get_num_gpus") else 0
            if n_gpus > 0 and hasattr(faiss, "StandardGpuResources"):
                res = faiss.StandardGpuResources()
                return faiss.index_cpu_to_gpu(res, 0, cpu_index)
        except Exception as e:
            print(f"FAISS GPU indisponible ({e}) — index CPU.")
    return cpu_index


def uses_e5_format(model_name):
    return "e5" in model_name.lower()


def prepare_for_embedding(texts, model_name, kind):
    texts = [clean_text(t) for t in texts]
    if uses_e5_format(model_name):
        prefix = "query: " if kind == "query" else "passage: "
        return [prefix + t for t in texts]
    return texts


def encode_texts(model, texts, model_name, kind="passage", batch_size=BATCH_SIZE):
    prepared = prepare_for_embedding(texts, model_name=model_name, kind=kind)
    embeddings = model.encode(
        prepared,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return np.asarray(embeddings, dtype="float32")


def search_index(query_embedding, index, df, top_k=5):
    query_embedding = np.asarray(query_embedding, dtype="float32").reshape(1, -1)
    scores, indices = index.search(query_embedding, top_k)
    results = df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    results["rank"]  = list(range(1, len(results) + 1))
    return results.reset_index(drop=True)

## 4. Téléchargement rapide via les fichiers XML GitHub (recommandé)

L'API Scaife ATLAS impose une requête HTTP par passage, ce qui devient très long.
Perseus publie en réalité ses textes sous forme de **fichiers TEI XML uniques par édition** sur GitHub :

- `PerseusDL/canonical-greekLit` (Homère, Platon, Thucydide, etc.)
- `OpenGreekAndLatin/First1KGreek` (textes complémentaires)

L'URN encode directement le chemin du fichier :
`urn:cts:greekLit:tlg0012.tlg001.perseus-grc2` → `data/tlg0012/tlg001/tlg0012.tlg001.perseus-grc2.xml`

On parse le XML pour extraire les unités de base (vers chez Homère, paragraphes chez les prosateurs)
et on les regroupe en passages d'environ `GITHUB_GROUP_TARGET_WORDS` mots.

In [4]:
import xml.etree.ElementTree as ET

PERSEUS_REPOS = [
    ("PerseusDL/canonical-greekLit",      "master"),
    ("OpenGreekAndLatin/First1KGreek",    "master"),
]


def urn_to_relative_path(edition_urn):
    last = edition_urn.split(":")[-1]
    textgroup, work, _ = last.split(".", 2)
    return f"data/{textgroup}/{work}/{last}.xml"


def fetch_tei_xml(edition_urn, timeout=30):
    rel = urn_to_relative_path(edition_urn)
    last_url = None
    for repo, branch in PERSEUS_REPOS:
        url = f"https://raw.githubusercontent.com/{repo}/{branch}/{rel}"
        last_url = url
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and r.text.strip().startswith("<"):
                return r.text, url
        except Exception:
            continue
    raise FileNotFoundError(f"XML introuvable pour {edition_urn} (dernier essai : {last_url})")


def parse_tei_leaves(xml_text):
    for tag in ("note", "bibl", "teiHeader"):
        xml_text = re.sub(rf"<{tag}\b[^>]*>.*?</{tag}>", "", xml_text, flags=re.DOTALL)
    xml_text = re.sub(r'\sxmlns="[^"]+"', "", xml_text, count=1)

    root = ET.fromstring(xml_text)

    parent_of = {child: parent for parent in root.iter() for child in parent}

    all_tps = [e for e in root.iter() if e.tag == "div" and e.get("type") == "textpart"]
    leaf_tps = [
        tp for tp in all_tps
        if not any(e.tag == "div" and e.get("type") == "textpart"
                   for e in tp.iter() if e is not tp)
    ]

    leaves = []
    for tp in leaf_tps:
        ref_parts, node = [], tp
        while node is not None:
            if node.tag == "div" and node.get("type") == "textpart":
                ref_parts.insert(0, node.get("n", "?"))
            node = parent_of.get(node)
        text = " ".join("".join(tp.itertext()).split())
        if text and ref_parts:
            leaves.append((".".join(ref_parts), text))
    return leaves


def group_leaves_by_words(leaves, target_words=200, max_leaf_words=None):
    if max_leaf_words is None:
        max_leaf_words = 2 * target_words

    split = []
    for ref, text in leaves:
        words = text.split()
        if len(words) <= max_leaf_words:
            split.append((ref, text))
        else:
            for i in range(0, len(words), target_words):
                sub = " ".join(words[i:i + target_words])
                split.append((f"{ref}#{i // target_words + 1}", sub))

    chunks, current, n_words = [], [], 0
    for ref, text in split:
        current.append((ref, text))
        n_words += len(text.split())
        if n_words >= target_words:
            chunks.append(current)
            current, n_words = [], 0
    if current:
        chunks.append(current)
    return chunks


def build_corpus_from_github_xml(editions, cache_path, target_words=200):
    cache_path = Path(cache_path)
    if cache_path.exists():
        existing = pd.read_csv(cache_path)
        done_urns = set(existing["edition_urn"].dropna().unique())
        print(f"Cache GitHub existant : {len(existing)} passages, {len(done_urns)} éditions.")
    else:
        existing, done_urns = pd.DataFrame(), set()

    for edition in editions:
        urn = edition["edition_urn"]
        if urn in done_urns:
            print(f"[skip] {edition.get('author','')} — {edition.get('title','')} (déjà en cache)")
            continue
        try:
            xml_text, url = fetch_tei_xml(urn)
        except Exception as e:
            print(f"  FAIL {urn} : {e}")
            continue
        try:
            leaves = parse_tei_leaves(xml_text)
        except ET.ParseError as e:
            print(f"  Parsing XML échoué pour {urn} : {e}")
            continue
        chunks = group_leaves_by_words(leaves, target_words=target_words)
        print(f"  OK {edition.get('title','')}: {len(leaves)} unités -> {len(chunks)} chunks ({url})")

        rows = []
        for block in chunks:
            ref_first, ref_last = block[0][0], block[-1][0]
            reference = ref_first if ref_first == ref_last else f"{ref_first}-{ref_last}"
            rows.append({
                "chunk_id":     f"{urn}:{reference}",
                "source":       "PerseusDL XML (GitHub)",
                "edition_urn":  urn,
                "reference":    reference,
                "author":       edition.get("author", ""),
                "title":        edition.get("title", ""),
                "language":     edition.get("language", ""),
                "text_greek":   " ".join(b[1] for b in block),
                "text_fr_hint": "",
                "pair_ids":     "",
            })

        existing = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
        existing.to_csv(cache_path, index=False)

    if existing.empty:
        raise RuntimeError("Aucun texte récupéré depuis GitHub.")
    return existing

## 5. Charger les paires alignées grec–français

Le fichier `train.csv` contient au minimum les colonnes `greek` et `french`.
Les autres (`greek_length`, `french_length`) sont ignorées.

In [5]:
aligned_path = find_existing_path(ALIGNED_PAIRS_PATH)
if aligned_path is None:
    raise FileNotFoundError(
        f"Impossible de trouver {ALIGNED_PAIRS_PATH}. Téléversez train.csv dans /content."
    )

pairs_df = pd.read_csv(aligned_path)
print(f"Fichier chargé : {aligned_path}")
print(f"Dimensions brutes : {pairs_df.shape}")
print("Colonnes :", pairs_df.columns.tolist())

assert GREEK_PAIR_COLUMN  in pairs_df.columns, f"Colonne manquante : {GREEK_PAIR_COLUMN}"
assert FRENCH_PAIR_COLUMN in pairs_df.columns, f"Colonne manquante : {FRENCH_PAIR_COLUMN}"

pairs_df = pairs_df[[GREEK_PAIR_COLUMN, FRENCH_PAIR_COLUMN]].copy()
pairs_df = pairs_df.rename(columns={GREEK_PAIR_COLUMN: "greek", FRENCH_PAIR_COLUMN: "french"})
pairs_df["pair_id"] = [f"pair_{i:06d}" for i in range(len(pairs_df))]
pairs_df["greek"]   = pairs_df["greek"].apply(clean_text)
pairs_df["french"]  = pairs_df["french"].apply(clean_text)
pairs_df = pairs_df[pairs_df["greek"].astype(bool) & pairs_df["french"].astype(bool)].reset_index(drop=True)

pairs_df["greek_n_words"]  = pairs_df["greek"].str.split().str.len()
pairs_df["french_n_words"] = pairs_df["french"].str.split().str.len()

print(f"Dimensions après nettoyage : {pairs_df.shape}")
display(pairs_df.head())
display(pairs_df[["greek_n_words", "french_n_words"]].describe())

Fichier chargé : train.csv
Dimensions brutes : (19371, 4)
Colonnes : ['greek', 'french', 'greek_length', 'french_length']
Dimensions après nettoyage : (19371, 5)


,greek,french,pair_id,greek_n_words,french_n_words
0,« τοὺς ἐχθρούς σου,« les ennemis de toi,pair_000000,4,5
1,καθὼς εἴρηκεν αὐτοῖς·,selon qu’il avait dit à eux ;,pair_000001,3,7
2,τοσαύτης ἀδικίας,d'une si-grande injustice,pair_000002,2,3
3,τῷ δὲ πέμπτῳ ἄρα,et le cinquième jour donc,pair_000003,4,5
4,Ἀναφλύστιος εἶπεν·,d'Anaphlyste a dit :,pair_000004,2,4


,greek_n_words,french_n_words
count,19371.000000,19371.000000
mean,3.012132,4.099788
std,1.157859,1.601273
min,1.000000,1.000000
25%,2.000000,3.000000
50%,3.000000,4.000000
75%,4.000000,5.000000
max,8.000000,11.000000


### 6. Construire ou charger le corpus grec

Ordre de préférence :

1. **A.** corpus grec externe via `GREEK_CORPUS_PATH` si défini (CSV avec `text_greek`/`greek`/`text`, ou TXT) ;
2. **B.** GitHub XML (rapide, défaut) si `USE_GITHUB_XML_FIRST` ;
3. **C.** Scaife ATLAS (lent, fallback) si GitHub échoue et `USE_SCAIFE_ATLAS_CORPUS` ;
4. **D.** corpus construit à partir des paires alignées (dépannage).

In [6]:
def build_corpus_from_pairs(pairs_df, group_size=8):
    rows = []
    for start in range(0, len(pairs_df), group_size):
        block = pairs_df.iloc[start:start + group_size]
        rows.append({
            "chunk_id":     f"aligned_chunk_{start:06d}_{start + len(block) - 1:06d}",
            "source":       "aligned_pairs",
            "edition_urn":  "",
            "reference":    f"paires {start}-{start + len(block) - 1}",
            "author":       "",
            "title":        "",
            "language":     "grc",
            "text_greek":   " ".join(block["greek"].tolist()),
            "text_fr_hint": " ".join(block["french"].tolist()),
            "pair_ids":     " ".join(block["pair_id"].tolist()),
        })
    return pd.DataFrame(rows)


def load_external_greek_corpus(path):
    path = find_existing_path(path)
    if path is None:
        raise FileNotFoundError(f"Corpus grec introuvable : {path}")
    if path.suffix.lower() == ".txt":
        raw    = path.read_text(encoding="utf-8")
        chunks = chunk_words(raw, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS)
        return pd.DataFrame([
            {"chunk_id": f"txt_chunk_{i:06d}", "source": path.name, "edition_urn": "",
             "reference": f"chunk {i}", "author": "", "title": "", "language": "grc",
             "text_greek": chunk, "text_fr_hint": "", "pair_ids": ""}
            for i, chunk in enumerate(chunks)
        ])
    df = pd.read_csv(path)
    text_col = next((c for c in ["text_greek", "greek", "text"] if c in df.columns), None)
    if text_col is None:
        raise ValueError("Le CSV doit contenir 'text_greek', 'greek' ou 'text'.")
    rows = []
    for i, row in df.iterrows():
        raw_text = clean_text(row[text_col])
        chunks   = chunk_words(raw_text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS)
        for j, chunk in enumerate(chunks):
            row_id = row.get("id", f"row_{i:06d}")
            rows.append({
                "chunk_id":     row_id if len(chunks) == 1 else f"{row_id}_chunk_{j:03d}",
                "source":       row.get("source", path.name),
                "edition_urn":  row.get("edition_urn", ""),
                "reference":    row.get("reference", row.get("title", f"ligne {i}")),
                "author":       row.get("author", ""),
                "title":        row.get("title", ""),
                "language":     row.get("language", "grc"),
                "text_greek":   chunk,
                "text_fr_hint": row.get("text_fr", ""),
                "pair_ids":     "",
            })
    return pd.DataFrame(rows)


if GREEK_CORPUS_PATH:
    corpus_df = load_external_greek_corpus(GREEK_CORPUS_PATH)
    print(f"Corpus grec externe chargé : {GREEK_CORPUS_PATH}")

elif USE_GITHUB_XML_FIRST:
    try:
        corpus_df = build_corpus_from_github_xml(
            SCAIFE_EDITIONS,
            cache_path=GITHUB_CACHE_PATH,
            target_words=GITHUB_GROUP_TARGET_WORDS,
        )
        print(f"Corpus GitHub XML prêt — cache : {GITHUB_CACHE_PATH}")
    except Exception as e:
        print(f"GitHub XML a échoué ({e}).")



else:
    corpus_df = build_corpus_from_pairs(pairs_df, group_size=GROUP_SIZE_FOR_RAG)
    print("Corpus grec construit à partir des paires alignées.")

for col in ["chunk_id", "source", "edition_urn", "reference", "author", "title",
            "language", "text_greek", "text_fr_hint", "pair_ids"]:
    if col not in corpus_df.columns:
        corpus_df[col] = ""

corpus_df["text_greek"] = corpus_df["text_greek"].apply(clean_text)
corpus_df = corpus_df[corpus_df["text_greek"].astype(bool)].reset_index(drop=True)
corpus_df = maybe_limit_df(corpus_df, MAX_CORPUS_CHUNKS)

print(f"Nombre de passages grecs : {len(corpus_df)}")
display(corpus_df[["chunk_id", "author", "title", "reference", "text_greek"]].head())
display(corpus_df.groupby(["author", "title"]).size().reset_index(name="n_passages"))

Cache GitHub existant : 1916 passages, 4 éditions.
[skip] Homerus — Ilias (déjà en cache)
[skip] Homerus — Odyssea (déjà en cache)
[skip] Thucydides — Historiae (déjà en cache)
[skip] Plato — Respublica (déjà en cache)
Corpus GitHub XML prêt — cache : github_greek_corpus.csv
Nombre de passages grecs : 1916


,chunk_id,author,title,reference,text_greek
0,urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1#1,Homerus,Ilias,1#1,"μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος οὐλομένην, ἣ..."
1,urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1#2,Homerus,Ilias,1#2,γῆρας ἔπεισιν ἡμετέρῳ ἐνὶ οἴκῳ ἐν Ἄργεϊ τηλόθι...
2,urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1#3,Homerus,Ilias,1#3,"οὖν ἤγερθεν ὁμηγερέες τε γένοντο, τοῖσι δʼ ἀνι..."
3,urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1#4,Homerus,Ilias,1#4,σὺ δὲ φράσαι εἴ με σαώσεις. τὸν δʼ ἀπαμειβόμεν...
4,urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1#5,Homerus,Ilias,1#5,"ἑκηβόλος ἄλγεα τεύχει, οὕνεκʼ ἐγὼ κούρης Χρυση..."


,author,title,n_passages
0,Homerus,Ilias,549
1,Homerus,Odyssea,425
2,Plato,Respublica,274
3,Thucydides,Historiae,668


## 7. Charger les modèles d'embeddings

- **modèle multilingue généraliste** (LaBSE) pour la baseline français → grec ;
- **modèle spécialisé grec/latin** (SPhilBerta) pour représenter les passages grecs dans la stratégie avec pont bilingue.

Si le modèle spécialisé ne se charge pas, on bascule automatiquement sur le multilingue.

In [7]:
def load_sentence_model(model_name, fallback_name=None):
    try:
        print(f"Chargement : {model_name}")
        model = SentenceTransformer(model_name, device=DEVICE)
        print(f"  -> OK sur {DEVICE}")
        return model, model_name
    except Exception as e:
        print(f"  -> échec : {e}")
        if fallback_name is None:
            raise
        print(f"Bascule vers : {fallback_name}")
        model = SentenceTransformer(fallback_name, device=DEVICE)
        return model, fallback_name


cross_model, cross_model_name = load_sentence_model(CROSSLINGUAL_MODEL_NAME)

if GREEK_MODEL_NAME == cross_model_name:
    greek_model, greek_model_name = cross_model, cross_model_name
else:
    greek_model, greek_model_name = load_sentence_model(GREEK_MODEL_NAME, fallback_name=cross_model_name)

Chargement : sentence-transformers/LaBSE


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> OK sur cuda
Chargement : bowphs/SPhilBerta


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bowphs/SPhilBerta
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> OK sur cuda


## 8. Stratégie A — baseline directe français → grec

On encode la question française et les passages grecs avec le même modèle multilingue.
C'est la baseline : pas besoin des paires alignées.

In [8]:
print("Encodage du corpus grec (modèle multilingue)...")
direct_corpus_embeddings = encode_texts(
    cross_model,
    corpus_df["text_greek"].tolist(),
    model_name=cross_model_name,
    kind="passage",
    batch_size=BATCH_SIZE,
)
direct_index = build_faiss_ip_index(direct_corpus_embeddings)


def direct_search_fr_to_greek(question_fr, top_k=TOP_K):
    query_embedding = encode_texts(
        cross_model, [question_fr], model_name=cross_model_name, kind="query", batch_size=1,
    )[0]
    return search_index(query_embedding, direct_index, corpus_df, top_k=top_k)

Encodage du corpus grec (modèle multilingue)...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

## 9. Stratégie B — pont bilingue (paires alignées)

1. On cherche les `ANCHOR_K` paires françaises les plus proches de la question.
2. On encode leurs équivalents grecs avec le modèle grec et on moyenne les embeddings → requête grecque enrichie.
3. On interroge le corpus grec encodé avec le modèle grec.

In [9]:
anchor_pairs_df = maybe_limit_df(pairs_df, MAX_ANCHOR_PAIRS)
print(f"Paires utilisées comme ancres : {len(anchor_pairs_df)}")

print("Encodage des phrases françaises alignées...")
french_anchor_embeddings = encode_texts(
    cross_model, anchor_pairs_df["french"].tolist(),
    model_name=cross_model_name, kind="passage", batch_size=BATCH_SIZE,
)
french_anchor_index = build_faiss_ip_index(french_anchor_embeddings)

print("Encodage du corpus grec (modèle spécialisé)...")
greek_corpus_embeddings = encode_texts(
    greek_model, corpus_df["text_greek"].tolist(),
    model_name=greek_model_name, kind="passage", batch_size=BATCH_SIZE,
)
greek_index = build_faiss_ip_index(greek_corpus_embeddings)


def find_french_anchors(question_fr, anchor_k=ANCHOR_K):
    q = encode_texts(cross_model, [question_fr], model_name=cross_model_name, kind="query", batch_size=1)[0]
    return search_index(q, french_anchor_index, anchor_pairs_df, top_k=anchor_k)


def build_greek_query_from_anchors(anchors_df):
    embeds = encode_texts(
        greek_model, anchors_df["greek"].tolist(),
        model_name=greek_model_name, kind="query",
        batch_size=min(BATCH_SIZE, len(anchors_df)),
    )
    return l2_normalize_vector(embeds.mean(axis=0))


def bridge_search_fr_to_greek(question_fr, top_k=TOP_K, anchor_k=ANCHOR_K):
    anchors = find_french_anchors(question_fr, anchor_k=anchor_k)
    q       = build_greek_query_from_anchors(anchors)
    return search_index(q, greek_index, corpus_df, top_k=top_k), anchors

Paires utilisées comme ancres : 19371
Encodage des phrases françaises alignées...


Batches:   0%|          | 0/606 [00:00<?, ?it/s]

Encodage du corpus grec (modèle spécialisé)...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

## 10. Stratégie C — recherche directe dans les paires alignées

Sert de contrôle : on regarde quelles paires alignées (et donc quels passages grecs courts) ressemblent le plus à la question française.

In [10]:
def aligned_pair_search(question_fr, top_k=TOP_K):
    results = find_french_anchors(question_fr, anchor_k=top_k).copy()
    results["text_greek"]   = results["greek"]
    results["text_fr_hint"] = results["french"]
    results["author"]       = "(paire alignée)"
    results["title"]        = ""
    results["reference"]    = results["pair_id"]
    return results

## 11. Comparer les trois stratégies sur une question

In [11]:
QUESTION = "colère"

direct_results              = direct_search_fr_to_greek(QUESTION, top_k=TOP_K)
bridge_results, anchors_df  = bridge_search_fr_to_greek(QUESTION, top_k=TOP_K, anchor_k=ANCHOR_K)
pair_results                = aligned_pair_search(QUESTION, top_k=TOP_K)

print("QUESTION :", QUESTION)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION : colère


In [12]:
def print_results(title, results_df, include_french_hint=False, max_chars=700):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    for _, row in results_df.iterrows():
        print(f"\n[{int(row['rank'])}] score={row['score']:.3f} | {row.get('author','')} {row.get('title','')} {row.get('reference','')}")
        greek = row["text_greek"]
        print(greek[:max_chars] + ("..." if len(greek) > max_chars else ""))
        if include_french_hint and "text_fr_hint" in row and clean_text(row["text_fr_hint"]):
            print("\nIndice français aligné :")
            hint = row["text_fr_hint"]
            print(hint[:max_chars] + ("..." if len(hint) > max_chars else ""))


print_results("A. Baseline directe français -> grec", direct_results)
print_results("B. Pont bilingue par paires alignées",  bridge_results, include_french_hint=True)
print_results("C. Recherche directe dans les paires",  pair_results)

print("\nAncres françaises utilisées pour la stratégie B :")
display(anchors_df[["rank", "score", "french", "greek"]])


A. Baseline directe français -> grec

[1] score=0.300 | Homerus Odyssea 18#7
ἔπος τʼ ἔφατʼ ἔκ τʼ ὀνόμαζεν· Εὐρυνόμη, θυμός μοι ἐέλδεται, οὔ τι πάρος γε, μνηστήρεσσι φανῆναι, ἀπεχθομένοισί περ ἔμπης· παιδὶ δέ κεν εἴποιμι ἔπος, τό κε κέρδιον εἴη, μὴ πάντα μνηστῆρσιν ὑπερφιάλοισιν ὁμιλεῖν, οἵ τʼ εὖ μὲν βάζουσι, κακῶς δʼ ὄπιθεν φρονέουσι. τὴν δʼ αὖτʼ Εὐρυνόμη ταμίη πρὸς μῦθον ἔειπεν· ναὶ δὴ ταῦτά γε πάντα, τέκος, κατὰ μοῖραν ἔειπες. ἀλλʼ ἴθι καὶ σῷ παιδὶ ἔπος φάο μηδʼ ἐπίκευθε, χρῶτʼ ἀπονιψαμένη καὶ ἐπιχρίσασα παρειάς· μηδʼ οὕτω δακρύοισι πεφυρμένη ἀμφὶ πρόσωπα ἔρχευ, ἐπεὶ κάκιον πενθήμεναι ἄκριτον αἰεί. ἤδη μὲν γάρ τοι παῖς τηλίκος, ὃν σὺ μάλιστα ἠρῶ ἀθανάτοισι γενειήσαντα ἰδέσθαι. τὴν δʼ αὖτε προσέειπε περίφρων Πηνελόπεια· Εὐρυνόμη, μὴ ταῦτα παραύδα, κηδομένη περ, χρῶτ...

[2] score=0.260 | Thucydides Historiae 4.43.2-4.44.2
καὶ πρῶτα μὲν τῷ δεξιῷ κέρᾳ τῶν Ἀθηναίων εὐθὺς ἀποβεβηκότι πρὸ τῆς Χερσονήσου οἱ Κορίνθιοι ἐπέκειντο, ἔπειτα δὲ καὶ τῷ ἄλλῳ στρατεύματι. καὶ ἦν ἡ μάχη καρτερὰ καὶ ἐ

,rank,score,french,greek
0,1,0.727299,étant en haine,ὄντες ἐν ἔχθρᾳ
1,2,0.718602,s’étant irrité,ὀργισθεὶς
2,3,0.651967,furent remplis de colère.,ἐπλήσθησαν θυμοῦ.


## 12. Charger Qwen pour la génération

Qwen2.5-7B-Instruct gère bien le français **et** le grec ancien dans le contexte.
On le charge en **4 bits** (bitsandbytes) pour tenir sur un GPU Colab T4 (16 Go).

Si vous avez plus de mémoire (A100, etc.), mettez `QWEN_LOAD_IN_4BIT = False` en haut.
Si vous avez moins, remplacez par `Qwen/Qwen2.5-3B-Instruct`.

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)

model_kwargs = {
    "torch_dtype": "auto",
    "device_map":  "auto",
}

if QWEN_LOAD_IN_4BIT and DEVICE == "cuda":
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    print("Qwen chargé en 4 bits (bitsandbytes).")
elif DEVICE != "cuda":
    print("Pas de GPU détecté.")

qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_NAME, **model_kwargs)
qwen_model.eval()
print("Qwen prêt :", QWEN_MODEL_NAME)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen prêt : Qwen/Qwen2.5-3B-Instruct


## 13. Construire le prompt RAG

On formate les passages grecs récupérés en un contexte numéroté `[P1] … [P2] …`,
qu'on injecte dans un prompt système + utilisateur.
Qwen doit répondre en français et **citer** les passages par leur numéro.

Quand l'indice français aligné existe (corpus construit à partir des paires), on le donne aussi :
cela aide énormément le modèle quand le grec est difficile.

In [14]:
SYSTEM_PROMPT = (
    "Tu es un assistant spécialisé en grec ancien et en philologie classique. "
    "On te fournit une question en français et un ensemble de passages en grec ancien "
    "numérotés [P1], [P2], etc., parfois accompagnés d'un indice de traduction française. "
    "Réponds en français, de manière concise et précise, en t'appuyant uniquement sur les "
    "passages fournis. Cite les passages utilisés entre crochets, par exemple [P1] ou [P2, P3]. "
    "Si les passages ne permettent pas de répondre, dis-le clairement plutôt que d'inventer."
)


def format_passages_for_prompt(retrieved_df, max_chars_per_passage=900, include_french_hint=True):
    blocks = []
    for _, row in retrieved_df.iterrows():
        head = f"[P{int(row['rank'])}] {row.get('author','')} — {row.get('title','')} ({row.get('reference','')})"
        greek = clean_text(row["text_greek"])
        if len(greek) > max_chars_per_passage:
            greek = greek[:max_chars_per_passage] + "..."
        block = f"{head}\nGrec : {greek}"
        if include_french_hint:
            hint = clean_text(row.get("text_fr_hint", ""))
            if hint:
                if len(hint) > max_chars_per_passage:
                    hint = hint[:max_chars_per_passage] + "..."
                block += f"\nIndice français aligné : {hint}"
        blocks.append(block)
    return "\n\n".join(blocks)


def build_rag_messages(question_fr, retrieved_df, include_french_hint=True):
    context = format_passages_for_prompt(retrieved_df, include_french_hint=include_french_hint)
    user_msg = (
        f"Question : {question_fr}\n\n"
        f"Passages :\n\n{context}\n\n"
        "Donne ta réponse en français, en citant les passages utilisés avec [P1], [P2]…"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]

## 14. Génération avec Qwen

In [15]:
@torch.inference_mode()
def generate_with_qwen(messages, max_new_tokens=QWEN_MAX_NEW_TOKENS,
                       temperature=QWEN_TEMPERATURE, top_p=QWEN_TOP_P):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

    do_sample = temperature is not None and temperature > 0
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"]       = top_p

    out_ids = qwen_model.generate(**inputs, **gen_kwargs)
    gen_ids = out_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

## 15. Pipeline unifié : `ask_qwen`

Une fonction qui prend une question française et :

1. lance le retrieval (stratégie au choix : `"bridge"`, `"direct"`, `"pair"`) ;
2. affiche les passages retenus ;
3. construit le prompt RAG et appelle Qwen ;
4. renvoie la réponse + les passages utilisés (pour audit).

In [16]:
def retrieve(question_fr, strategy="bridge", top_k=TOP_K, anchor_k=ANCHOR_K):
    if strategy == "direct":
        return direct_search_fr_to_greek(question_fr, top_k=top_k), None
    if strategy == "pair":
        return aligned_pair_search(question_fr, top_k=top_k), None
    if strategy == "bridge":
        return bridge_search_fr_to_greek(question_fr, top_k=top_k, anchor_k=anchor_k)
    raise ValueError(f"Stratégie inconnue : {strategy}")


def ask_qwen(question_fr, strategy="bridge", top_k=TOP_K, anchor_k=ANCHOR_K,
             include_french_hint=True, show_passages=True, max_new_tokens=QWEN_MAX_NEW_TOKENS):
    retrieved, anchors = retrieve(question_fr, strategy=strategy, top_k=top_k, anchor_k=anchor_k)

    if show_passages:
        print("=" * 100)
        print(f"Question : {question_fr}")
        print(f"Stratégie : {strategy} | top_k={top_k} | anchor_k={anchor_k}")
        print("=" * 100)
        for _, row in retrieved.iterrows():
            print(f"\n[P{int(row['rank'])}] score={row['score']:.3f} | {row.get('author','')} {row.get('title','')} {row.get('reference','')}")
            print(textwrap.shorten(row["text_greek"], width=400, placeholder="..."))
            if include_french_hint and clean_text(row.get("text_fr_hint", "")):
                print("  FR aligné :", textwrap.shorten(row["text_fr_hint"], width=400, placeholder="..."))
        if anchors is not None:
            print("\nAncres françaises (stratégie bridge) :")
            for _, row in anchors.iterrows():
                print(f"  [{int(row['rank'])}] score={row['score']:.3f} | FR: {row['french']} | GR: {row['greek']}")
        print()

    messages = build_rag_messages(question_fr, retrieved, include_french_hint=include_french_hint)
    answer   = generate_with_qwen(messages, max_new_tokens=max_new_tokens)

    print("=" * 100)
    print("Réponse Qwen :")
    print("=" * 100)
    print(answer)

    return {"question": question_fr, "strategy": strategy, "answer": answer,
            "passages": retrieved, "anchors": anchors}

## 16. Exemples

Modifiez la question, comparez les stratégies, regardez ce que Qwen cite.

In [17]:
result = ask_qwen(
    "Que dit Homère sur la colère d'Achille ?",
    strategy="bridge",
    top_k=5,
    anchor_k=3,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question : Que dit Homère sur la colère d'Achille ?
Stratégie : bridge | top_k=5 | anchor_k=3

[P1] score=0.804 | Plato Respublica 3.387
καὶ τὸ— ψυχὴ δὲ κατὰ χθονός, ἠΰτε καπνός, ᾤχετο τετριγυῖα καὶ— ὡς δʼ ὅτε νυκτερίδες μυχῷ ἄντρου θεσπεσίοιο τρίζουσαι ποτέονται, ἐπεί κέ τις ἀποπέσῃσιν ὁρμαθοῦ ἐκ πέτρης, ἀνά τʼ ἀλλήλῃσιν ἔχονται, ὣς αἳ τετριγυῖαι ἅμʼ ᾔεσαν. ταῦτα καὶ τὰ τοιαῦτα πάντα παραιτησόμεθα Ὅμηρόν τε καὶ τοὺς ἄλλους ποιητὰς μὴ χαλεπαίνειν ἂν διαγράφωμεν, οὐχ ὡς οὐ ποιητικὰ καὶ ἡδέα τοῖς πολλοῖς ἀκούειν, ἀλλʼ ὅσῳ...

[P2] score=0.779 | Homerus Ilias 13#30-14#1
φαίδιμος Ἕκτωρ· Αἶαν ἁμαρτοεπὲς βουγάϊε ποῖον ἔειπες· εἰ γὰρ ἐγὼν οὕτω γε Διὸς πάϊς αἰγιόχοιο εἴην ἤματα πάντα, τέκοι δέ με πότνια Ἥρη, τιοίμην δʼ ὡς τίετʼ Ἀθηναίη καὶ Ἀπόλλων, ὡς νῦν ἡμέρη ἥδε κακὸν φέρει Ἀργείοισι πᾶσι μάλʼ, ἐν δὲ σὺ τοῖσι πεφήσεαι, αἴ κε ταλάσσῃς μεῖναι ἐμὸν δόρυ μακρόν, ὅ τοι χρόα λειριόεντα δάψει· ἀτὰρ Τρώων κορέεις κύνας ἠδʼ οἰωνοὺς δημῷ καὶ σάρκεσσι πεσὼν ἐπὶ νηυσὶν...

[P3] score=0.778 | Homerus Od

In [18]:
result = ask_qwen(
    "Quelle est la définition platonicienne de la justice dans la République ?",
    strategy="direct",
    top_k=5,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question : Quelle est la définition platonicienne de la justice dans la République ?
Stratégie : direct | top_k=5 | anchor_k=3

[P1] score=0.457 | Plato Respublica 8.562
τί οὖν; τετάχθω ἡμῖν κατὰ δημοκρατίαν ὁ τοιοῦτος ἀνήρ, ὡς δημοκρατικὸς ὀρθῶς ἂν προσαγορευόμενος; τετάχθω, ἔφη. ἡ καλλίστη δή, ἦν δʼ ἐγώ, πολιτεία τε καὶ ὁ κάλλιστος ἀνὴρ λοιπὰ ἂν ἡμῖν εἴη διελθεῖν, τυραννίς τε καὶ τύραννος. κομιδῇ γʼ, ἔφη. φέρε δή, τίς τρόπος τυραννίδος, ὦ φίλε ἑταῖρε, γίγνεται; ὅτι μὲν γὰρ ἐκ δημοκρατίας μεταβάλλει σχεδὸν δῆλον. δῆλον. ἆρʼ οὖν τρόπον τινὰ τὸν αὐτὸν ἔκ τε...

[P2] score=0.381 | Plato Respublica 8.545
διεληλύθαμεν. ἆρʼ οὖν τὸ μετὰ τοῦτο διιτέον τοὺς χείρους, τὸν φιλόνικόν τε καὶ φιλότιμον, κατὰ τὴν Λακωνικὴν ἑστῶτα πολιτείαν, καὶ ὀλιγαρχικὸν αὖ καὶ δημοκρατικὸν καὶ τὸν τυραννικόν, ἵνα τὸν ἀδικώτατον ἰδόντες ἀντιθῶμεν τῷ δικαιοτάτῳ καὶ ἡμῖν τελέα ἡ σκέψις ᾖ, πῶς ποτε ἡ ἄκρατος δικαιοσύνη πρὸς ἀδικίαν τὴν ἄκρατον ἔχει εὐδαιμονίας τε πέρι τοῦ ἔχοντος καὶ ἀθλιότητος, ἵνα ἢ Θρασυμάχῳ πειθόμ

In [19]:
# Pour comparer rapidement les deux stratégies sur la même question :
for strat in ["direct", "bridge"]:
    print("\n" + "#" * 100)
    print("# Stratégie :", strat)
    print("#" * 100)
    ask_qwen("Comment Thucydide caractérise-t-il la guerre civile ?", strategy=strat, top_k=5)


####################################################################################################
# Stratégie : direct
####################################################################################################


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question : Comment Thucydide caractérise-t-il la guerre civile ?
Stratégie : direct | top_k=5 | anchor_k=3

[P1] score=0.367 | Plato Respublica 8.562
τί οὖν; τετάχθω ἡμῖν κατὰ δημοκρατίαν ὁ τοιοῦτος ἀνήρ, ὡς δημοκρατικὸς ὀρθῶς ἂν προσαγορευόμενος; τετάχθω, ἔφη. ἡ καλλίστη δή, ἦν δʼ ἐγώ, πολιτεία τε καὶ ὁ κάλλιστος ἀνὴρ λοιπὰ ἂν ἡμῖν εἴη διελθεῖν, τυραννίς τε καὶ τύραννος. κομιδῇ γʼ, ἔφη. φέρε δή, τίς τρόπος τυραννίδος, ὦ φίλε ἑταῖρε, γίγνεται; ὅτι μὲν γὰρ ἐκ δημοκρατίας μεταβάλλει σχεδὸν δῆλον. δῆλον. ἆρʼ οὖν τρόπον τινὰ τὸν αὐτὸν ἔκ τε...

[P2] score=0.364 | Plato Respublica 5.468
τί δὲ δή, εἶπον, τὰ περὶ τὸν πόλεμον; πῶς ἑκτέον σοι τοὺς στρατιώτας πρὸς αὑτούς τε καὶ τοὺς πολεμίους; ἆρʼ ὀρθῶς μοι καταφαίνεται ἢ οὔ; λέγʼ, ἔφη, ποῖʼ αὖ. αὐτῶν μέν, εἶπον, τὸν λιπόντα τάξιν ἢ ὅπλα ἀποβαλόντα ἤ τι τῶν τοιούτων ποιήσαντα διὰ κάκην ἆρα οὐ δημιουργόν τινα δεῖ καθιστάναι ἢ γεωργόν; πάνυ μὲν οὖν. τὸν δὲ ζῶντα εἰς τοὺς πολεμίους ἁλόντα ἆρʼ οὐ δωρεὰν διδόναι τοῖς ἑλοῦσι χρῆσθαι τῇ...

[P3] score=

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question : Comment Thucydide caractérise-t-il la guerre civile ?
Stratégie : bridge | top_k=5 | anchor_k=3

[P1] score=0.803 | Thucydides Historiae 1.78.1-1.80.1
‘βουλεύεσθε οὖν βραδέως ὡς οὐ περὶ βραχέων, καὶ μὴ ἀλλοτρίαις γνώμαις καὶ ἐγκλήμασι πεισθέντες οἰκεῖον πόνον πρόσθησθε. τοῦ δὲ πολέμου τὸν παράλογον, ὅσος ἐστί, πρὶν ἐν αὐτῷ γενέσθαι προδιάγνωτε· μηκυνόμενος γὰρ φιλεῖ ἐς τύχας τὰ πολλὰ περιίστασθαι, ὧν ἴσον τε ἀπέχομεν καὶ ὁποτέρως ἔσται ἐν ἀδήλῳ κινδυνεύεται. ἰόντες τε οἱ ἄνθρωποι ἐς τοὺς πολέμους τῶν ἔργων πρότερον ἔχονται, ἃ χρῆν ὕστερον...

[P2] score=0.792 | Homerus Ilias 13#11
ἀρᾶται δὲ τάχιστα μιγήμεναι ἐν δαῒ λυγρῇ· οὐδέ κεν ἔνθα τεόν γε μένος καὶ χεῖρας ὄνοιτο. εἴ περ γάρ κε βλεῖο πονεύμενος ἠὲ τυπείης οὐκ ἂν ἐν αὐχένʼ ὄπισθε πέσοι βέλος οὐδʼ ἐνὶ νώτῳ, ἀλλά κεν ἢ στέρνων ἢ νηδύος ἀντιάσειε πρόσσω ἱεμένοιο μετὰ προμάχων ὀαριστύν. ἀλλʼ ἄγε μηκέτι ταῦτα λεγώμεθα νηπύτιοι ὣς ἑσταότες, μή πού τις ὑπερφιάλως νεμεσήσῃ· ἀλλὰ σύ γε κλισίην δὲ κιὼν ἕλευ ὄβριμον ἔγχος. ὣς...

[P